# Block 4 — the Hubbard atom from partial spectral functions

Numerical companion to

> F. B. Kugler, S.-S. B. Lee and J. von Delft,
> *Multipoint Correlation Functions: Spectral Representation and Numerical Evaluation*,
> [Phys. Rev. X **11**, 041006 (2021)](https://doi.org/10.1103/PhysRevX.11.041006)

The plan is to assemble the Matsubara correlators of the half-filled Hubbard atom
directly from their spectral representation, and to check them against the closed
forms the paper quotes:

| Step | Content | Benchmark |
|:--|:--|:--|
| **1** | spectrum: $H$, $d_\sigma$, $d^\dagger_\sigma$, $n_\sigma$; diagonalise; $\rho_i = e^{-\beta E_i}/Z$ | $Z = 2(1+e^{\beta u})$ by hand |
| **2** | partial spectral functions, Eq. (28), general in $\ell$ | — |
| **3** | the kernel, Eq. (46), two-branch form | finite by construction |
| **4** | $\ell = 2$: assemble Eq. (39) | $G(i\nu) = \tfrac12\sum_\pm (i\nu \pm U/2)^{-1}$, App. E |
| **5** | $\ell = 4$: 24 permutations, subtract Eq. (73), amputate Eq. (76) | Eqs. (85a), (85b) |
| **6** | null test at $U = 0$ | every connected object vanishes |

**All six steps are implemented and checked.** The test suite
(`julia --project test/runtests.jl`) carries 46,279 assertions.

Model: $H = \varepsilon_d (n_\uparrow + n_\downarrow) + U n_\uparrow n_\downarrow$
at $\varepsilon_d = -U/2$, on the four states
$|0\rangle, |\uparrow\rangle, |\downarrow\rangle, |\uparrow\downarrow\rangle$.
This is the $U/\Delta \to \infty$ limit of the symmetric AIM, Eq. (79).

In [1]:
include(joinpath(@__DIR__, "..", "src", "HubbardAtom.jl"))
using .HubbardAtom
using LinearAlgebra
using Printf

## Step 1 — spectrum

Build the operators with correct fermionic signs, diagonalise $H$, and form the
Boltzmann weights. Everything later hangs off the eigenbasis produced here:
Eq. (28) needs the matrix elements $(O)_{i,i+1}$ in the energy eigenbasis, which
`to_eigenbasis` supplies.

The operators come from a Jordan–Wigner transform with fermionic mode order
$(\uparrow, \downarrow)$,

$$ d_\uparrow = \mathbb{1} \otimes f, \qquad d_\downarrow = f \otimes z, \qquad
   f = \begin{pmatrix} 0 & 1 \\ 0 & 0\end{pmatrix}, \quad
   z = \begin{pmatrix} 1 & 0 \\ 0 & -1\end{pmatrix}, $$

so the parity string $z$ over the $\uparrow$ mode is what produces
$d_\downarrow |\uparrow\downarrow\rangle = -|\uparrow\rangle$. That sign is not
cosmetic: it feeds every permutation sign $\zeta_p$ in Eq. (39).

In [2]:
U, β = 2.0, 5.0
model = HubbardAtomModel(U, β)
ops = operators(model)

@printf("U = %.3f,  β = %.3f,  εd = %.3f,  u = U/2 = %.3f\n",
        model.U, model.β, level_position(model), half_interaction(model))
println("\nFock basis order: ", join(BASIS_LABELS, "  "))
println("\nH (Fock basis) =")
display(ops.H)

U = 2.000,  β = 5.000,  εd = -1.000,  u = U/2 = 1.000

Fock basis order: 

|0⟩  |↑⟩  |↓⟩  |↑↓⟩

H (Fock basis) =


4×4 Matrix{Float64}:
 0.0   0.0   0.0  0.0
 0.0  -1.0   0.0  0.0
 0.0   0.0  -1.0  0.0
 0.0   0.0   0.0  0.0

In [3]:
# Fermionic algebra: the checks that catch a wrong Jordan-Wigner sign.
I4 = Matrix(1.0I, DIM, DIM)

println("{d↑, d†↑} == 1        : ", ops.d_up * ops.dag_up + ops.dag_up * ops.d_up ≈ I4)
println("{d↓, d†↓} == 1        : ", ops.d_dn * ops.dag_dn + ops.dag_dn * ops.d_dn ≈ I4)
println("{d↑, d†↓} == 0        : ", ops.d_up * ops.dag_dn + ops.dag_dn * ops.d_up ≈ zeros(DIM, DIM))
println("{d↑, d↓}  == 0        : ", ops.d_up * ops.d_dn + ops.d_dn * ops.d_up ≈ zeros(DIM, DIM))
println("d↑² == d↓² == 0       : ", ops.d_up^2 ≈ zeros(DIM, DIM) && ops.d_dn^2 ≈ zeros(DIM, DIM))
println("[H, N] == 0           : ", ops.H * ops.N ≈ ops.N * ops.H)
println("[H, Sz] == 0          : ", ops.H * ops.Sz ≈ ops.Sz * ops.H)

# the fermionic sign itself
up_dn = [0.0, 0.0, 0.0, 1.0]
up    = [0.0, 1.0, 0.0, 0.0]
println("\nd↓|↑↓⟩ = -|↑⟩         : ", ops.d_dn * up_dn ≈ -up)
println("d↑|↑↓⟩ = +|↓⟩         : ", ops.d_up * up_dn ≈ [0.0, 0.0, 1.0, 0.0])

{d↑, d†↑} == 1        : true


{d↓, d†↓} == 1        : true
{d↑, d†↓} == 0        : true
{d↑, d↓}  == 0        : true
d↑² == d↓² == 0       : true


[H, N] == 0           : true
[H, Sz] == 0          : true

d↓|↑↓⟩ = -|↑⟩         : 

true
d↑|↑↓⟩ = +|↓⟩         : true


In [4]:
sp = spectrum(model)

println("eigenenergies   E = ", sp.E)
println("closed form       = ", eigenenergies_analytic(model))
println("agree             : ", sp.E ≈ eigenenergies_analytic(model))

println("\nH·V == V·diag(E)  : ", ops.H * sp.V ≈ sp.V * Diagonal(sp.E))
println("V orthonormal     : ", permutedims(sp.V) * sp.V ≈ Matrix(1.0I, DIM, DIM))

eigenenergies   E = 

[-1.0, -1.0, 0.0, 0.0]
closed form       = 

[-1.0, -1.0, 0.0, 0.0]
agree             : true

H·V == V·diag(E)  : true
V orthonormal     : 

true


In [5]:
# Check Z by hand.  With u = U/2 and εd = -U/2 the spectrum is
#     E(|0⟩) = 0,  E(|↑⟩) = E(|↓⟩) = -u,  E(|↑↓⟩) = 2εd + U = 0,
# so     Z = 1 + e^{βu} + e^{βu} + 1 = 2(1 + e^{βu}).
Z_hand = partition_function_analytic(model)

@printf("Z (diagonalisation) = %.15g\n", sp.Z)
@printf("Z (by hand)         = %.15g\n", Z_hand)
@printf("relative difference = %.3e\n", abs(sp.Z - Z_hand) / Z_hand)
println("agree to machine precision: ", isapprox(sp.Z, Z_hand; rtol = 1e-12))

Z (diagonalisation) = 298.826318205153
Z (by hand)         = 298.826318205153


relative difference = 1.902e-16
agree to machine precision: 

true


In [6]:
# Boltzmann weights and the quantum numbers of each eigenstate.
Nvals, Szvals = quantum_numbers(sp, ops)

println(" i      E_i        ρ_i          N      Sz")
println("-"^50)
for i in 1:DIM
    @printf("%2d  %9.5f  %11.4e  %5.1f  %6.1f\n", i, sp.E[i], sp.ρ[i], Nvals[i], Szvals[i])
end
@printf("\nΣ ρ_i = %.15g\n", sum(sp.ρ))

 i      E_i        ρ_i          N      Sz
--------------------------------------------------
 1   -1.00000   4.9665e-01    1.0    -0.5
 2   -1.00000   4.9665e-01    1.0     0.5


 3    0.00000   3.3464e-03    0.0     0.0
 4    0.00000   3.3464e-03    2.0     0.0

Σ ρ_i = 1


In [7]:
# Particle-hole symmetry at εd = -U/2 pins the atom to half filling.
@printf("⟨n↑⟩       = %.15g\n", thermal_average(sp, ops.n_up))
@printf("⟨n↓⟩       = %.15g\n", thermal_average(sp, ops.n_dn))
@printf("⟨N⟩        = %.15g\n", thermal_average(sp, ops.N))
@printf("⟨Sz⟩       = %.3e\n",  thermal_average(sp, ops.Sz))
@printf("⟨n↑n↓⟩     = %.15g   (exact: 1/Z = %.15g)\n",
        thermal_average(sp, ops.n_up * ops.n_dn), 1 / sp.Z)
@printf("⟨H⟩        = %.15g\n", thermal_average(sp, ops.H))

# tanh(βu/2) of Eq. (85) is this spectrum's polarisation
u = half_interaction(model)
singly = 2 * exp(β * u) / Z_hand
@printf("\n2·P(singly occupied) - 1 = %.15g\ntanh(βu/2)               = %.15g\n",
        2 * singly - 1, tanh(β * u / 2))

⟨n↑⟩       = 0.5
⟨n↓⟩       = 0.5


⟨N⟩        = 1
⟨Sz⟩       = 0.000e+00
⟨n↑n↓⟩     = 0.00334642546214243   (exact: 1/Z = 0.00334642546214243)
⟨H⟩        = -0.993307149075715

2·P(singly occupied) - 1 = 0.98661429815143
tanh(βu/2)               = 0.98661429815143


In [8]:
# What Step 2 will consume: operators in the energy eigenbasis, Eq. (28).
d_rot = to_eigenbasis(sp, ops.d_up)
println("d↑ in the energy eigenbasis:")
display(round.(d_rot, digits = 12))
println("\nnonzero elements (i, j, value), each lowering N by one:")
for j in 1:DIM, i in 1:DIM
    if abs(d_rot[i, j]) > 1e-10
        @printf("  (%d, %d)  % .6f    N: %.1f → %.1f\n",
                i, j, d_rot[i, j], Nvals[j], Nvals[i])
    end
end

d↑ in the energy eigenbasis:


4×4 Matrix{Float64}:
 0.0  0.0  0.0  1.0
 0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0
 0.0  0.0  0.0  0.0


nonzero elements (i, j, value), each lowering N by one:
  (3, 2)   1.000000    N: 1.0 → 0.0
  (1, 4)   1.000000    N: 2.0 → 1.0


In [9]:
# Z against its closed form across a parameter sweep, including U = 0 and U < 0.
println("    U      β        Z (diag)          Z (hand)        rel.err")
println("-"^66)
for (Ui, βi) in [(1.0, 1.0), (2.0, 5.0), (4.0, 0.3), (0.0, 2.0), (8.0, 10.0), (-2.0, 1.5)]
    mi = HubbardAtomModel(Ui, βi)
    spi = spectrum(mi)
    Zi = partition_function_analytic(mi)
    @printf("%6.1f %6.1f  %16.8e  %16.8e  %10.2e\n",
            Ui, βi, spi.Z, Zi, abs(spi.Z - Zi) / Zi)
end

    U      β        Z (diag)          Z (hand)        rel.err
------------------------------------------------------------------
   1.0    1.0    5.29744254e+00    5.29744254e+00    1.68e-16
   2.0    5.0    2.98826318e+02    2.98826318e+02    1.90e-16


   4.0    0.3    5.64423760e+00    5.64423760e+00    0.00e+00
   0.0    2.0    4.00000000e+00    4.00000000e+00    0.00e+00
   8.0   10.0    4.70770534e+17    4.70770534e+17    0.00e+00
  -2.0    1.5    2.44626032e+00    2.44626032e+00    0.00e+00


## Step 2 — partial spectral functions

Eq. (28), implemented literally and kept general in $\ell$, so the one function
serves $\ell = 2$ (Step 4), $\ell = 3$ and $\ell = 4$ (Step 5):

$$ S[\mathcal{O}_p](\omega'_p) = \sum_{1 \cdots \ell} \rho_1
   \Big[ \prod_{i=1}^{\ell-1} (O_{\bar{i}})_{i,i+1}\,
   \delta\big(\omega'_{\bar{1}\cdots\bar{i}} - E_{i+1,1}\big) \Big]
   (O_{\bar{\ell}})_{\ell,1}, \qquad E_{i+1,1} = E_{i+1} - E_1 . $$

`psf` returns one `PSFTerm` per eigenstate cycle with nonvanishing weight, each carrying

- **`position`** — the $\ell-1$ partial sums $\omega'_{\bar{1}\cdots\bar{i}} = E_{i+1} - E_1$
  at which the deltas sit. These are precisely the arguments the Eq. (46) kernel
  consumes, through $\Omega_{\bar{1}\cdots\bar{i}} = i\omega_{\bar{1}\cdots\bar{i}} - \omega'_{\bar{1}\cdots\bar{i}}$.
- **`weight`** — $\rho_1 \prod_{i=1}^{\ell} (O_{\bar{i}})_{i,i+1}$, the product taken
  *cyclically* so the last factor is $(O_{\bar{\ell}})_{\ell,1}$. This is exactly the
  numerator of Eq. (29), which is what Eqs. (26)–(28) collapse to.

Operators are handed over in the permuted order $\mathcal{O}_p = (O_{\bar 1},\dots,O_{\bar\ell})$,
so one call covers one permutation $p$; Steps 4 and 5 loop over permutations and pair each
`position` with the matching $i\omega_{\bar{1}\cdots\bar{i}}$.

Because the PSFs are sums of $\delta$'s, the $\omega'$ integral in Eq. (39) will be a
**finite sum** over these terms — no grids, no broadening.

In [10]:
# ℓ = 2, the simplest case: S[d↑, d†↑].
terms2 = psf(sp, ops.d_up, ops.dag_up)
println("raw terms: ", length(terms2), "   (one per eigenstate cycle with nonzero weight)\n")
println("    ω'          weight        eigenstate cycle")
println("-"^50)
for t in terms2
    @printf("%8.3f   %12.6e   %s\n", t.position[1], real(t.weight), string(t.states))
end

# Merging coincident deltas gives the PSF as an actual measure.
println("\naggregated:")
for t in aggregate_psf(terms2)
    @printf("%8.3f   %12.6e\n", t.position[1], real(t.weight))
end

raw terms: 2

   (one per eigenstate cycle with nonzero weight)

    ω'          weight        eigenstate cycle
--------------------------------------------------
  -1.000   3.346425e-03   [3, 2]
   1.000   4.966536e-01   [1, 4]



aggregated:
  -1.000   3.346425e-03
   1.000   4.966536e-01


In [11]:
# Closed form for this PSF. d↑ connects |↑⟩→|0⟩ and |↑↓⟩→|↓⟩, so the deltas sit
# at ω' = ∓u with weights ρ|0⟩ = 1/Z and ρ|↓⟩ = e^{βu}/Z.
agg2 = aggregate_psf(terms2)
byω = Dict(round(t.position[1]; digits = 9) => real(t.weight) for t in agg2)

@printf("weight at ω' = -u = %5.2f :  %.15g   (exact 1/Z      = %.15g)\n",
        -u, byω[round(-u; digits = 9)], 1 / sp.Z)
@printf("weight at ω' = +u = %5.2f :  %.15g   (exact e^{βu}/Z = %.15g)\n",
         u, byω[round(u; digits = 9)], exp(β * u) / sp.Z)

weight at ω' = -u = -1.00 :  0.00334642546214243   (exact 1/Z      = 0.00334642546214243)
weight at ω' = +u =  1.00 :  0.496653574537858   (exact e^{βu}/Z = 0.496653574537857)


In [12]:
# Sum rule. Summing Eq. (28) over every eigenstate cycle collapses the cyclic
# product to a trace:
#     Σ weights = tr(ρ O_1̄ O_2̄ ⋯ O_ℓ̄) = ⟨O_1̄ O_2̄ ⋯ O_ℓ̄⟩ ,
# the equal-time expectation value of the operator product in the same order.
# This holds for every ℓ, and is the main check on Step 2.
labelled = [
    ("(d↑, d†↑)",              (ops.d_up, ops.dag_up)),
    ("(d†↑, d↑)",              (ops.dag_up, ops.d_up)),
    ("(n↑, n↓)",               (ops.n_up, ops.n_dn)),
    ("(n↑, d↑, d†↑)",          (ops.n_up, ops.d_up, ops.dag_up)),
    ("(d↑, n↑, d†↑)",          (ops.d_up, ops.n_up, ops.dag_up)),
    ("(d↑, d†↑, d↑, d†↑)",     (ops.d_up, ops.dag_up, ops.d_up, ops.dag_up)),
    ("(d↑, d†↑, d↓, d†↓)",     (ops.d_up, ops.dag_up, ops.d_dn, ops.dag_dn)),
    ("(d↑, d†↓, d↓, d†↑)",     (ops.d_up, ops.dag_dn, ops.d_dn, ops.dag_up)),
]

println(" ℓ   operator tuple           terms    Σ weights        ⟨O_1̄⋯O_ℓ̄⟩         |diff|")
println("-"^82)
for (name, Os) in labelled
    terms = psf(sp, Os)
    lhs = real(psf_total_weight(terms))
    rhs = thermal_average(sp, reduce(*, Os))
    @printf(" %d   %-22s %5d  %14.10f  %14.10f   %8.1e\n",
            length(Os), name, length(terms), lhs, rhs, abs(lhs - rhs))
end

 ℓ   operator tuple           terms    Σ weights        ⟨O_1̄⋯O_ℓ̄⟩         |diff|
----------------------------------------------------------------------------------
 2   (d↑, d†↑)                  2    0.5000000000    0.5000000000    0.0e+00
 2   (d†↑, d↑)                  2    0.5000000000    0.5000000000    0.0e+00


 2   (n↑, n↓)                   1    0.0033464255    0.0033464255    0.0e+00
 3   (n↑, d↑, d†↑)              0    0.0000000000    0.0000000000    0.0e+00


 3   (d↑, n↑, d†↑)              2    0.5000000000    0.5000000000    0.0e+00
 4   (d↑, d†↑, d↑, d†↑)         2    0.5000000000    0.5000000000    0.0e+00


 4   (d↑, d†↑, d↓, d†↓)         1    0.0033464255    0.0033464255    0.0e+00
 4   (d↑, d†↓, d↓, d†↑)         1    0.4966535745    0.4966535745    0.0e+00


In [13]:
# A first look ahead to Step 4. The local spectral function is
#     A(ω) = S[d,d†](ω) + S[d†,d](-ω) ,
# and Appendix E's exact propagator G(iν) = ½ Σ± (iν ± U/2)⁻¹ says it must be
# two poles of weight ½ at ω = ±u -- with no temperature dependence at all,
# even though each PSF separately is strongly β dependent.
A = Dict{Float64, Float64}()
for t in aggregate_psf(psf(sp, ops.d_up, ops.dag_up))
    k = round(t.position[1]; digits = 9)
    A[k] = get(A, k, 0.0) + real(t.weight)
end
for t in aggregate_psf(psf(sp, ops.dag_up, ops.d_up))
    k = round(-t.position[1]; digits = 9)      # note the reflection ω → -ω
    A[k] = get(A, k, 0.0) + real(t.weight)
end

println("    ω        A(ω)      expected")
println("-"^34)
for k in sort(collect(keys(A)))
    @printf("%8.3f  %9.6f   %9.6f\n", k, A[k], 0.5)
end
@printf("\nΣ A(ω) = %.15g   (sum rule ⟨{d, d†}⟩ = 1)\n", sum(values(A)))

    ω        A(ω)      expected
----------------------------------
  -1.000   0.500000    0.500000
   1.000   0.500000    0.500000



Σ A(ω) = 1   (sum rule ⟨{d, d†}⟩ = 1)


In [14]:
# ℓ = 4: the object Step 5 will need, here for one of the 24 permutations.
Os4 = (ops.d_up, ops.dag_up, ops.d_dn, ops.dag_dn)
terms4 = psf(sp, Os4)

println("ℓ = 4, permutation (d↑, d†↑, d↓, d†↓)")
println("raw terms: ", length(terms4), "   distinct positions: ", length(aggregate_psf(terms4)))
println("\n   ω'₁      ω'₁₂     ω'₁₂₃        weight       cycle")
println("-"^62)
for t in terms4
    @printf("%7.3f  %7.3f  %7.3f   %12.6e   %s\n",
            t.position[1], t.position[2], t.position[3], real(t.weight), string(t.states))
end

# The individual frequencies behind those partial sums conserve energy.
for t in terms4
    ω = psf_frequencies(t)
    @printf("\nω' = %s\n  Σω' = %.2e  (energy conservation ω'_1̄⋯ℓ̄ = 0)\n",
            string(round.(ω, digits = 6)), sum(ω))
end

ℓ = 4, permutation (d↑, d†↑, d↓, d†↓)
raw terms: 1   distinct positions: 1

   ω'₁      ω'₁₂     ω'₁₂₃        weight       cycle
--------------------------------------------------------------
 -1.000    0.000   -1.000   3.346425e-03   [3, 2, 3, 1]



ω' = [-1.0, 1.0, -1.0, 1.0]
  Σω' = 0.00e+00  (energy conservation ω'_1̄⋯ℓ̄ = 0)


## Step 3 — the kernel

Eq. (46), the two-branch form, with an explicit test for a vanishing composite
$\Omega_{\bar{1}\cdots\bar{j}}$:

$$ K(\Omega_p) = \begin{cases}
\displaystyle\prod_{i=1}^{\ell-1} \Omega^{-1}_{\bar{1}\cdots\bar{i}},
& \text{if } \prod_{i=1}^{\ell-1} \Omega_{\bar{1}\cdots\bar{i}} \neq 0, \\[2ex]
-\dfrac{1}{2}\Big[\beta + \displaystyle\sum_{\substack{i=1 \\ i \neq j}}^{\ell-1}
\Omega^{-1}_{\bar{1}\cdots\bar{i}}\Big]
\displaystyle\prod_{\substack{i=1 \\ i \neq j}}^{\ell-1}
\Omega^{-1}_{\bar{1}\cdots\bar{i}},
& \text{if } \Omega_{\bar{1}\cdots\bar{j}} = 0,
\end{cases} $$

with $\Omega_{\bar{1}\cdots\bar{i}} = i\omega_{\bar{1}\cdots\bar{i}} - \omega'_{\bar{1}\cdots\bar{i}}$.
Deliberately **not** Eq. (45) with an $\epsilon$-regulator: Eq. (46) never forms a
divergent factor at all — the vanishing one is *excluded* from both the sum and the
product — so it is finite by construction.

**One design decision.** $\Omega$ vanishes only when *both* parts do: the Matsubara
part $\omega_{\bar{1}\cdots\bar{j}}$ and, through a degeneracy $E_{j+1} = E_{\bar 1}$,
the spectral part $\omega'_{\bar{1}\cdots\bar{j}}$. The Matsubara part is exactly zero
or not at all, so `MatsubaraFreq` stores the **integer** $m$ in $\omega = m\pi/\beta$ and
the test is an integer comparison. Float-comparing $\Omega$ to zero would confuse a
genuinely small frequency — $\pi/\beta$ is tiny at large $\beta$ — with a composite that
is identically zero and needs the anomalous branch.

In [15]:
# Regular branch: a plain product of inverse composites.
Ωreg = ComplexF64[1.5 + 0.5im, -2.0 + 1.0im, 0.25 - 3.0im]
println("regular   K = ", matsubara_kernel(Ωreg, [false, false, false], β))
println("  ∏ Ω⁻¹     = ", prod(inv, Ωreg), "\n")

# Anomalous branch, ℓ = 2: empty product and empty sum leave K = -β/2.
println("ℓ=2 anomalous K = ", matsubara_kernel(ComplexF64[0.0], [true], β),
        "   (-β/2 = ", -β/2, ")\n")

# Anomalous branch, ℓ = 4 with j = 2. This is literally the second part of
# Eq. (B26): -½[β + 1/Ω_1̄ + 1/Ω_1̄2̄3̄] / (Ω_1̄ Ω_1̄2̄3̄).
Ωan = ComplexF64[1.5 + 0.5im, 0.0, -0.75 + 2.0im]
K   = matsubara_kernel(Ωan, [false, true, false], β)
B26 = -0.5 * (β + inv(Ωan[1]) + inv(Ωan[3])) * inv(Ωan[1]) * inv(Ωan[3])
println("ℓ=4 anomalous K = ", K)
println("Eq. (B26)       = ", B26)
println("agree           : ", K ≈ B26)
println("finite          : ", isfinite(real(K)) && isfinite(imag(K)),
        "   (no 1/0 is ever formed)")

regular   K = 0.005517241379310348

 - 0.09379310344827586im
  ∏ Ω⁻¹     = 0.005517241379310337 - 0.09379310344827586im

ℓ=2 anomalous K = -2.5 - 0.0im   (-β/2 = -2.5)

ℓ=4 anomalous K = 0.5797860761869017

 + 0.5660048789641584im
Eq. (B26)       = 0.5797860761869018 + 0.5660048789641583im
agree           : true
finite          : true   (no 1/0 is ever formed)


In [16]:
# Why the integer index matters. At β = 1e12 a genuinely nonzero bosonic
# frequency is numerically smaller than any sane float tolerance, yet Ω ≠ 0
# and the regular branch is the correct one.
βbig = 1e12
msbig = [MatsubaraFreq(1), MatsubaraFreq(1), MatsubaraFreq(-1), MatsubaraFreq(-1)]
Ωb, vanb = composites(msbig, [0.0, 0.0, 0.0], βbig; atol = 1e-10)
@printf("Ω₁₂    = %.3e   (|Ω₁₂| < atol = 1e-10)\n", abs(Ωb[2]))
println("flagged as vanishing? ", vanb[2], "   <- integer test says m₁₂ = ",
        msbig[1].m + msbig[2].m, " ≠ 0")
println("so the kernel takes the regular branch: ",
        matsubara_kernel(Ωb, vanb, βbig) ≈ prod(inv, Ωb))

Ω₁₂    = 6.283e-12   (|Ω₁₂| < atol = 1e-10)
flagged as vanishing? 

false   <- integer test says m₁₂ = 2 ≠ 0
so the kernel takes the regular branch: true


## Step 4 — first milestone, $\ell = 2$

Assemble Eq. (39),

$$ G(i\omega) = \sum_p \zeta_p \int d^{\ell-1}\omega'_p \;
   K(i\omega_p - \omega'_p)\, S[\mathcal{O}_p](\omega'_p), $$

for $\mathcal{O} = (d_\sigma, d^\dagger_\sigma)$ and compare against the exact
propagator of Appendix E,

$$ G(i\nu) = \frac{1}{2}\sum_{\pm} (i\nu \pm U/2)^{-1} = \frac{-i\nu}{\nu^2 + u^2}. $$

The sign $\zeta_p$ is $+1$ ($-1$) if $\mathcal{O}_p$ differs from $\mathcal{O}$ by an
even (odd) number of transpositions of **fermionic** operators — so it is not simply
the parity of $p$ when bosonic operators are present.

The paper's stop rule: do not proceed until this matches to machine precision.

In [17]:
println("  n |            G(iν) from Eq. (39)             |      exact (App. E)      | rel err")
println("-"^96)
for n in 0:5
    ν = fermionic_freq(n)
    got  = propagator(model, ν; sp = sp, ops = ops)
    want = propagator_exact(model, ν)
    @printf("%3d | %+.15f %+.15fim | %+.10f %+.10fim | %.2e\n",
            n, real(got), imag(got), real(want), imag(want), abs(got-want)/abs(want))
end

  n |            G(iν) from Eq. (39)             |      exact (App. E)      | rel err
------------------------------------------------------------------------------------------------
  0 | +0.000000000000000 -0.450477243368389im | +0.0000000000 -0.4504772434im | 1.23e-16
  1 | +0.000000000000000 -0.413997749252169im | +0.0000000000 -0.4139977493im | 0.00e+00


  2 | +0.000000000000000 -0.289025482222236im | +0.0000000000 -0.2890254822im | 1.92e-16
  3 | +0.000000000000000 -0.216188454372683im | +0.0000000000 -0.2161884544im | 1.28e-16
  4 | +0.000000000000000 -0.171476420146551im | +0.0000000000 -0.1714764201im | 1.62e-16
  5 | +0.000000000000000 -0.141719536856365im | +0.0000000000 -0.1417195369im | 3.92e-16


In [18]:
# The milestone, swept: every parameter set, n = -8 … 8.
println("    U      β   | max rel err |G_num - G_exact|/|G_exact|  over n = -8…8")
println("-"^72)
for (Ui, βi) in [(1.0,1.0), (2.0,5.0), (4.0,0.3), (0.0,2.0), (8.0,10.0), (-2.0,1.5)]
    mi = HubbardAtomModel(Ui, βi); spi = spectrum(mi); opsi = operators(mi)
    worst = 0.0
    for n in -8:8
        ν = fermionic_freq(n)
        g = propagator(mi, ν; sp = spi, ops = opsi); e = propagator_exact(mi, ν)
        worst = max(worst, abs(g - e) / abs(e))
    end
    @printf("%6.1f %6.1f   |  %.3e\n", Ui, βi, worst)
end

    U      β   | max rel err |G_num - G_exact|/|G_exact|  over n = -8…8
------------------------------------------------------------------------
   1.0    1.0   |  4.364e-16
   2.0    5.0   |  3.917e-16


   4.0    0.3   |  2.545e-16
   0.0    2.0   |  1.962e-16
   8.0   10.0   |  2.314e-16
  -2.0    1.5   |  1.792e-16


## Step 5 — $\ell = 4$

$\mathcal{O} = (d_\sigma, d^\dagger_\sigma, d_{\sigma'}, d^\dagger_{\sigma'})$ over all
24 permutations. Subtract the disconnected part, Eq. (73),

$$ G^{\mathrm{dis}}_{\sigma\sigma'} = \beta\, G(i\omega_1) G(i\omega_3)
   \big(\delta_{\sigma\sigma'}\delta_{\omega_{23},0} - \delta_{\omega_{12},0}\big), $$

then amputate the external legs, Eq. (74),

$$ F_{\sigma\sigma'} = \frac{G^{\mathrm{con}}_{\sigma\sigma'}(i\omega_1,i\omega_2,i\omega_3)}
   {G(i\omega_1) G(-i\omega_2) G(i\omega_3) G(-i\omega_4)}, \qquad \omega_4 = -\omega_{123}. $$

Appendix D does **not** enter here: its imaginary frequency shifts are needed only for
the Keldysh amputation of Eq. (76), where the legs sit at real frequencies. On the
Matsubara axis the legs are exactly on discrete frequencies and there is nothing to shift.

### A correction to Eq. (85)

today.pdf warned that the text layer mangles Eq. (85) and to read it off the page image.
Transcribed from the text layer, the anomalous term reads

$$ \beta u^2 \frac{\delta_{\omega_{12}}\mathrm{th} + \delta_{\omega_{13}}(\mathrm{th}-1)
   + \delta_{\omega_{14}}(\mathrm{th}+1)}{\prod_i (i\omega_i + u)\prod_i(i\omega_i)}
   \qquad\text{(dimension energy}^{-7}\text{: wrong)} $$

which cannot be right — $\beta u^2$ has dimension of energy, so the remaining factor must
be dimensionless, while that denominator carries energy$^8$. The extraction dropped a
fraction bar and printed both products on one line. The factor is
$\prod_i(i\omega_i+u) \big/ \prod_i(i\omega_i)$, giving

$$ F_{\uparrow\downarrow} = 2u + u^3\frac{\sum_i (i\omega_i)^2}{\prod_i (i\omega_i)}
   - \frac{6u^5}{\prod_i (i\omega_i)}
   + \beta u^2 \big[\delta_{\omega_{12}}\mathrm{th} + \delta_{\omega_{13}}(\mathrm{th}-1)
   + \delta_{\omega_{14}}(\mathrm{th}+1)\big]
   \frac{\prod_i(i\omega_i+u)}{\prod_i(i\omega_i)}, $$

$$ F_{\uparrow\uparrow} = \beta u^2 \big(\delta_{\omega_{14}} - \delta_{\omega_{12}}\big)
   \frac{\prod_i(i\omega_i+u)}{\prod_i(i\omega_i)}, $$

with $u = U/2$, $\mathrm{th} = \tanh(\beta u/2)$. Three independent things confirm it:
dimensional consistency; Appendix E's separately stated second-order expansion
$F_{\uparrow\downarrow} = U + \tfrac14\beta U^2(\delta_{\omega_{14}} - \delta_{\omega_{13}})$
and $F_{\uparrow\uparrow} = \tfrac14\beta U^2(\delta_{\omega_{14}} - \delta_{\omega_{12}})$;
and a brute-force $\tau$-integration of Eq. (71) that reproduces the computed
$G^{(4)}$ independently of this whole machinery.

Note that whenever any $\delta$ fires the frequencies are $\pm$-paired,
$(\nu,-\nu,\nu',-\nu')$, so $\prod_i(i\omega_i+u) = \prod_i(i\omega_i-u)$ and the sign of
$u$ in that product is not observable.

In [19]:
# Both channels at a handful of points: generic ones, and ones where a δ fires.
println(" n1 n2 n3 | δ12 δ13 δ14 |   F↑↓ computed    F↑↓ exact   |   F↑↑ computed    F↑↑ exact")
println("-"^104)
for (n1, n2, n3) in [(0,0,0), (3,0,-2), (1,-1,0), (1,-2,0), (0,0,-1), (0,-1,0)]
    ms = vertex_frequencies(n1, n2, n3)
    d(k) = ms[1].m + ms[k].m == 0 ? 1 : 0
    fud, eud = vertex(model, :up, :dn, ms; sp=sp, ops=ops), vertex_exact(model, :updn, ms)
    fuu, euu = vertex(model, :up, :up, ms; sp=sp, ops=ops), vertex_exact(model, :upup, ms)
    @printf("%3d%3d%3d |%3d%4d%4d | %13.6f %13.6f | %13.6f %13.6f\n",
            n1,n2,n3, d(2),d(3),d(4), real(fud),real(eud), real(fuu),real(euu))
end

 n1 n2 n3 | δ12 δ13 δ14 |   F↑↓ computed    F↑↓ exact   |   F↑↑ computed    F↑↑ exact
--------------------------------------------------------------------------------------------------------
  0  0  0 |  0   0   0 |     24.964596     24.964596 |     -0.000000     -0.000000
  3  0 -2 |  0   0   0 |     -0.393066     -0.393066 |      0.000000      0.000000


  1 -1  0 |  0   0   1 |     37.064469     37.064469 |     22.636964     22.636964
  1 -2  0 |  1   0   0 |     14.427505     14.427505 |    -22.636964    -22.636964
  0  0 -1 |  0   1   1 |     76.522586     76.522586 |     62.411490     62.411490
  0 -1  0 |  1   0   1 |    138.934076    138.934076 |      0.000000      0.000000


In [20]:
# Swept over n1,n2,n3 ∈ -3:3 (343 points). Where F is identically zero we
# require it numerically, measured against the scale of F on the grid.
println("    U      β  | chan | max rel err (|F|>0) | max |F_num| where F ≡ 0 | scale of F")
println("-"^90)
for (Ui, βi) in [(1.0,1.0), (2.0,5.0), (4.0,0.3), (8.0,10.0), (-2.0,1.5)]
    mi = HubbardAtomModel(Ui, βi); spi = spectrum(mi); opsi = operators(mi)
    for (sa, sb, tag) in [(:up,:dn,:updn), (:up,:up,:upup)]
        re = 0.0; ze = 0.0; scale = 0.0
        for n1 in -3:3, n2 in -3:3, n3 in -3:3
            ms = vertex_frequencies(n1, n2, n3)
            got = vertex(mi, sa, sb, ms; sp=spi, ops=opsi)
            want = vertex_exact(mi, tag, ms)
            scale = max(scale, abs(want))
            abs(want) > 1e-6 ? (re = max(re, abs(got-want)/abs(want))) :
                               (ze = max(ze, abs(got)))
        end
        @printf("%6.1f %6.1f | %4s |      %.3e      |       %.3e        | %.3e\n",
                Ui, βi, tag, re, ze, scale)
    end
end

    U      β  | chan | max rel err (|F|>0) | max |F_num| where F ≡ 0 | scale of F
------------------------------------------------------------------------------------------
   1.0    1.0 | updn |      1.238e-13      |       0.000e+00        | 1.371e+00
   1.0    1.0 | upup |      7.981e-13      |       1.115e-14        | 2.628e-01


   2.0    5.0 | updn |      5.606e-14      |       0.000e+00        | 1.389e+02
   2.0    5.0 | upup |      4.764e-15      |       3.911e-15        | 6.241e+01
   4.0    0.3 | updn |      5.238e-14      |       0.000e+00        | 5.893e+00
   4.0    0.3 | upup |      3.397e-13      |       7.888e-14        | 1.289e+00


   8.0   10.0 | updn |      8.721e-16      |       0.000e+00        | 1.214e+07
   8.0   10.0 | upup |      5.038e-16      |       1.673e-11        | 4.257e+06
  -2.0    1.5 | updn |      2.131e-14      |       0.000e+00        | 5.911e+00
  -2.0    1.5 | upup |      4.312e-14      |       5.746e-15        | 2.262e+00


## Step 6 — null test

Set $U = 0$. Every connected object must vanish identically. This exercises all 24
permutations and every sign $\zeta_p$ against a target of exactly zero.

It also drives the anomalous branch of Eq. (46) hard: at $U = 0$ the whole spectrum is
degenerate, so $\omega' = 0$ for every PSF term and *every* vanishing bosonic composite
triggers the second branch. A sign error there would show up here.

In [21]:
println("  β   | max |G^con|          | max |F|              | max |G^dis| (nonzero!)")
println("-"^76)
for βi in (0.5, 2.0, 7.0)
    m0 = HubbardAtomModel(0.0, βi); sp0 = spectrum(m0); ops0 = operators(m0)
    gc = 0.0; fv = 0.0; gd = 0.0
    for n1 in -3:3, n2 in -3:3, n3 in -3:3, ch in ((:up,:dn), (:up,:up))
        ms = vertex_frequencies(n1, n2, n3)
        gc = max(gc, abs(connected_4p(m0, ch[1], ch[2], ms; sp=sp0, ops=ops0)))
        fv = max(fv, abs(vertex(m0, ch[1], ch[2], ms; sp=sp0, ops=ops0)))
        gd = max(gd, abs(disconnected_4p(m0, ch[1], ch[2], ms; sp=sp0, ops=ops0)))
    end
    @printf("%5.1f |  %.3e          |  %.3e          |  %.3e\n", βi, gc, fv, gd)
end
println("\nG^dis is large, so G^con = G - G^dis vanishing is a real cancellation,")
println("not a case of both sides being zero.")

  β   | max |G^con|          | max |F|              | max |G^dis| (nonzero!)
----------------------------------------------------------------------------
  0.5 |  4.337e-19          |  2.737e-14          |  1.267e-02
  2.0 |  2.776e-17          |  6.844e-15          |  8.106e-01


  7.0 |  1.807e-15          |  1.656e-14          |  3.475e+01

G^dis is large, so G^con = G - G^dis vanishing is a real cancellation,


not a case of both sides being zero.
